## Imports

In [2]:
import os
import uuid
import asyncio
import dotenv
import openai
import pandas as pd
from datetime import datetime, timezone
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import instructor

dotenv.load_dotenv()

True

## RAG Pipeline

In [ ]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field("Answer to the question")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "cm_interventions"


def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve(query, top_k=5):
    vector = embed_text(query)
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector,
        limit=top_k,
    ).points
    return hits


def format_context(hits):
    parts = []
    for h in hits:
        p = h.payload
        parts.append(
            f"ID: {p.get('id')}\nMachine: {p.get('machine')}\n"
            f"Date: {p.get('date_start')}\nSummary: {p.get('summary')}"
        )
    return "\n\n---\n\n".join(parts)


RAG_PROMPT = """\
You are a maintenance assistant. Use the intervention records below to answer the question.
If the records do not contain relevant information, say \"I don't know\".
Be concise. Do not use markdown formatting.

Question: {question}

Records:
{context}
"""


def generate_answer(question, context):
    prompt = RAG_PROMPT.format(question=question, context=context)
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort={"effort"="none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):
    hits = retrieve(question, top_k)
    context = format_context(hits)
    answer = generate_answer(question, context)

    retrieved_ids = [h.payload.get('id') for h in hits]
    scores = [h.get('score') for h in hits]

    final_result = {
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_ids,
        "retrieved_context": context,
        "similarity_score": scores
    }
    
    return final_answer

/Users/jooaobrum/Library/CloudStorage/GoogleDrive-joao.paulo.brum14@gmail.com/My Drive/Projetos Pessoais/Projetos de Estudo/end2end-ai-engineering-bootcamp/hephaestus-agentic-maintenance/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
